# SWPS · Analiza sesji sprzedażowych
Dane syntetyczne. 22.11.2026.

Cel: poprawny zbiór analityczny, eksploracja, paradoks Simpsona i granice wnioskowania. To nie jest eksperyment A/B.

Pliki CSV umieść obok notebooka. W Colab prześlij oba CSV przez panel plików. Wymagane pakiety: pandas, numpy, matplotlib. Nie potrzebujesz kluczy API.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
DATA = Path('.')
raw = pd.read_csv(DATA / 'sesje.csv')
customers_raw = pd.read_csv(DATA / 'klienci.csv')
print(raw.shape, customers_raw.shape)
print(raw.head().to_string(index=False))

## 1. Audyt · 10 min
Ustal jednostkę i klucz. Policz wiersze, unikalne sesje i duplikaty. Sprawdź typy i braki. Nie usuwaj sesji tylko dlatego, że brakuje wieku klienta.

In [ ]:
print('Wiersze sesji:', len(raw))
print('Unikalne sesje:', raw.session_id.nunique())
print('Dodatkowe kopie:', raw.duplicated('session_id').sum())
print('Słownik:', len(customers_raw), customers_raw.customer_id.nunique())
print(raw.dtypes)
print(customers_raw.isna().sum())
assert raw.loc[raw.duplicated('session_id', keep=False)].drop_duplicates().session_id.is_unique

## 2. Czyszczenie i join · 20 min
Zachowaj surową kwotę. Sprawdź identyczność kopii, usuń je po session_id. Parsuj przecinek dziesiętny i błędne teksty do NA. Najpierw zobacz błąd many_to_one. Następnie uporządkuj identyczne kopie klientów i wykonaj left join. Wymagane 850 sesji po połączeniu.

In [ ]:
sessions = raw.drop_duplicates().copy()
assert sessions.session_id.is_unique
sessions['order_value_raw'] = sessions.order_value_pln
sessions['order_value_pln'] = pd.to_numeric(sessions.order_value_raw.astype(str).str.replace(',', '.', regex=False), errors='coerce')
sessions['started_at'] = pd.to_datetime(sessions.started_at)
print('Nieznane kwoty:', sessions.order_value_pln.isna().sum())
try:
    sessions.merge(customers_raw, on='customer_id', how='left', validate='many_to_one')
except pd.errors.MergeError as e:
    print('Oczekiwany błąd słownika:', e)
customers = customers_raw.drop_duplicates().copy()
assert customers.customer_id.is_unique
data = sessions.merge(customers, on='customer_id', how='left', validate='many_to_one', indicator=True)
assert len(data) == 850 and data.session_id.is_unique
assert data['_merge'].eq('both').all()
assert data.loc[data.converted.eq(0), 'order_value_pln'].eq(0).all()
assert data.loc[data.converted.eq(1), 'order_value_pln'].dropna().gt(0).all()
print('Po join:', data.shape, data['_merge'].value_counts().to_dict())

## 3. Eksploracja · 15 min
Dla sesji z zakupem podaj liczbę znanych i nieznanych kwot, średnią, medianę i kwantyle. Pokaż histogram. Następnie oblicz konwersję kanałów razem z licznikami oraz mianownikami. Brak kwoty nie usuwa sesji z analizy konwersji.

In [ ]:
purchases = data.loc[data.converted.eq(1), 'order_value_pln']
print('Zakupy:', len(purchases), 'znane kwoty:', purchases.notna().sum(), 'brak kwoty:', purchases.isna().sum())
print(purchases.describe(percentiles=[.25,.5,.75,.95]).to_string())
purchases.dropna().hist(bins=35)
plt.xlabel('Wartość zamówienia [PLN]')
plt.ylabel('Liczba sesji z zakupem i znaną kwotą')
plt.title('Dane syntetyczne, n=' + str(purchases.notna().sum()))
plt.show()
overall = data.groupby('channel').agg(sessions=('session_id','size'), purchases=('converted','sum'))
overall['conversion'] = overall.purchases / overall.sessions
print(overall.to_string())

## 4. Paradoks Simpsona · 20 min
Oblicz konwersję według kanału i urządzenia. Wyznacz udziały urządzeń. Porównaj kanały przy tych samych wagach 50/50. Wyjaśnij, dlaczego wynik ogółem różni się od porównania w grupach. Nie interpretuj tego jako dowodu efektu przyczynowego.

In [ ]:
by_device = data.groupby(['channel','device']).agg(sessions=('session_id','size'),purchases=('converted','sum'))
by_device['conversion'] = by_device.purchases / by_device.sessions
print(by_device.to_string())
print(pd.crosstab(data.channel, data.device, normalize='index'))
rates = by_device.conversion.unstack('device')
standardized = rates.mul(pd.Series({'desktop':.5,'mobile':.5})).sum(axis=1)
print('Standaryzacja 50/50:', standardized.to_dict())
plot = rates.rename(columns={'desktop':'Desktop','mobile':'Mobile'}).copy()
plot['Ogółem'] = overall.conversion
plot.T.mul(100).plot.bar(rot=0,ylim=(0,100),color=['#e5b438','#317f9e'])
plt.ylabel('Konwersja [%]')
plt.title('Porównanie w grupach i ogółem, dane syntetyczne')
plt.show()
assert np.isclose(overall.loc['A','conversion'],315/480)
assert np.isclose(overall.loc['B','conversion'],263/370)
assert np.allclose(standardized.loc[['A','B']], [.75,.65])

## 5. Dostępność cech · 15 min
Prognoza powstaje na początku sesji. Oceń dostępność session_seconds, order_value_pln i refund_pln. Zaproponuj wcześniejszą historię klienta. Dla chętnych: wykonaj bootstrap klastrowy z kolejnego bloku.

In [ ]:
feature_review = pd.DataFrame([
 ['device','dostępne na wejściu, jeśli system je zna'],
 ['channel','dostępne na wejściu przy określonej atrybucji'],
 ['session_seconds','czas całej sesji niedostępny na wejściu'],
 ['order_value_pln','wynik zakupowy niedostępny na wejściu'],
 ['refund_pln','późniejsze zdarzenie niedostępne na wejściu'],
],columns=['kolumna','ocena'])
print(feature_review.to_string(index=False))
ordered = data.sort_values(['customer_id','started_at']).copy()
ordered['previous_sessions'] = ordered.groupby('customer_id').cumcount()
print(ordered[['customer_id','started_at','previous_sessions']].head(6).to_string(index=False))

## Rozszerzenie opcjonalne

In [ ]:
# Rozszerzenie: ilustracyjny bootstrap klastrowy.
# Jednostka resamplingu = klient. Estymand = różnica konwersji SESJI B-A.
# Założenie: klienci są niezależnymi klastrami. Nie naprawia obciążenia doboru.
stats = data.assign(A=data.channel.eq('A').astype(int), B=data.channel.eq('B').astype(int))
stats['yA'] = stats.A * stats.converted
stats['yB'] = stats.B * stats.converted
cluster = stats.groupby('customer_id')[['A','B','yA','yB']].sum().to_numpy()
generator = np.random.default_rng(22)
draws=[]
for _ in range(2000):
    totals = cluster[generator.integers(0,len(cluster),len(cluster))].sum(axis=0)
    if totals[0] > 0 and totals[1] > 0:
        draws.append(totals[3]/totals[1]-totals[2]/totals[0])
ci=np.quantile(draws,[.025,.975])
print('Różnica B-A:', (overall.loc['B','conversion']-overall.loc['A','conversion'])*100, 'pp')
print('Ilustracyjny przedział percentylowy 95% [pp]:',ci*100)
print('To nie jest przedział dla efektu przyczynowego ani dowód dotyczący realnej firmy.')

## Raport końcowy · 10 min
Oddaj tabelę kanał × urządzenie z licznikami i mianownikami, jeden wykres oraz cztery zdania:
1. Wynik ogółem.
2. Wynik w segmentach i wyjaśnienie różnicy.
3. Ograniczenie wnioskowania.
4. Uzasadniony następny krok.

Kryteria ćwiczenia (propozycja prowadzącego): jednostka i mianowniki 2 pkt, czyszczenie i join 2 pkt, proporcje i standaryzacja 3 pkt, granice wniosku i cech 2 pkt, czytelność 1 pkt.

### Przykładowy wniosek
B ma 71,08% wobec 65,63% w A ogółem. W obu grupach urządzeń A ma wynik wyższy o 10 pp, a odmienne udziały urządzeń odwracają agregat. Dane obserwacyjne bez kosztów i randomizacji nie ustalają efektu przesunięcia budżetu. Następny krok to ustalenie kosztów i marży oraz projektu porównania, który odpowie na pytanie o interwencję.